In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Training

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.2 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import torch
import json
from datetime import datetime
import os
import cv2
import numpy as np

torch.backends.cudnn.benchmark = True

# ==== 夜間前処理コールバック（学習=GPUガンマのみ / 評価=重い処理） ====
def build_night_preprocess_callback(
    enable_train: bool,
    enable_eval: bool,
    gamma: float = 0.6,
    clahe_clip: float = 3.0,
    clahe_grid: int = 8,
    denoise_h: int = 0,  # ← デフォは0（評価でだけ使うならOK）
):
    if not (enable_train or enable_eval):
        return None

    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(clahe_grid, clahe_grid))

    def _apply_cpu_hard_ops_rgb(img_uint8_rgb: np.ndarray) -> np.ndarray:
        # CLAHE + (任意)Denoise をCPUで
        lab = cv2.cvtColor(img_uint8_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        if denoise_h > 0:
            out = cv2.fastNlMeansDenoisingColored(out, None, denoise_h, denoise_h, 7, 21)
        return out

    def _callback(trainer):
        b = getattr(trainer, "batch", None)
        if b is None or "img" not in b:
            return
        imgs: torch.Tensor = b["img"]  # (N,3,H,W) float32 0-1
        is_training = bool(getattr(trainer, "training", True))

        if is_training:
            if not enable_train:
                return
            # === 学習: GPU上でガンマのみ ===
            # clampは安全のため。powはinplace避ける（AMP配慮）
            b["img"] = torch.clamp(imgs, 0, 1).pow(gamma)
            trainer.batch = b
        else:
            if not enable_eval:
                return
            # === 評価: 重い処理を1回だけ ===
            # ※ Val/Testは頻度が低いのでCPU往復を許容（速度影響は小）
            device = imgs.device
            np_imgs = (imgs.detach().cpu().numpy() * 255.0).astype(np.uint8)

            out = []
            for chw in np_imgs:
                hwc = np.transpose(chw, (1, 2, 0))                # CHW->HWC
                # 先にガンマ（GPU相当の見た目に合わせる）
                hwc = np.clip(np.power(hwc/255.0, gamma)*255.0, 0, 255).astype(np.uint8)
                hwc = _apply_cpu_hard_ops_rgb(hwc)                 # CLAHE(+Denoise)
                out.append(np.transpose(hwc.astype(np.float32)/255.0, (2,0,1)))
            b["img"] = torch.from_numpy(np.stack(out)).to(device)
            trainer.batch = b

    return _callback
# ==== settings ====
now_str = datetime.now().strftime('%Y%m%d_%H%M%S')
config = {
    "model": "yolov8n.pt",
    "data": "/content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/mixed_grobal/daynight.yaml",
    "epochs": 200,
    "imgsz": 320,
    "batch": 16,
    "name": f"furniture_yolov8n_{now_str}_all_data_Grobal_Histogram",
    "project": "/content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/results/furniture_project_3",
    "exist_ok": True,
    "device": "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "patience": 50,
    "workers": 8,
    "save": True,
    "save_period": -1,
    "cache": False,
    "close_mosaic": 0,
    "resume": False,
    "amp": True,
    "pretrained": True
}

# ==== Train ====
model = YOLO(config["model"])

# night_cb = build_night_preprocess_callback(
#     enable_train=True,   # 学習はGammaのみ（GPU）
#     enable_eval=True,    # Val/TestはGamma+CLAHE(+Denoise)を適用
#     gamma=0.6,
#     clahe_clip=3.0,
#     clahe_grid=8,
#     denoise_h=0,         # ★ 評価時だけ必要なら >0 に
# )
# if night_cb:
#     model.add_callback("on_preprocess_batch_end", night_cb)

results = model.train(**config)

# ==== Save ====
save_dir = model.trainer.save_dir
with open(os.path.join(save_dir, "config.json"), "w") as f:
    json.dump(config, f, indent=4)
print(f"✅ 訓練設定を保存しました: {os.path.join(save_dir, 'config.json')}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.187 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/mixed_grobal/daynight.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:850: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/200     0.586G          0      18.02          0          0        320: 100% ━━━━━━━━━━━━ 74/74 4.2it/s 17.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 3.1it/s 1.6s
                   all        146          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, can not compute metrics without labels


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:850: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/200     0.748G          0      11.64          0          0        320: 100% ━━━━━━━━━━━━ 74/74 4.2it/s 17.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 4.4it/s 1.1s
                   all        146          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, can not compute metrics without labels


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      5/200     0.783G          0      3.108          0          0        320: 100% ━━━━━━━━━━━━ 74/74 4.1it/s 18.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 3.1it/s 1.6s
                   all        146          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, can not compute metrics without labels


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:850: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      6/200     0.795G          0       1.24          0          0        320: 100% ━━━━━━━━━━━━ 74/74 4.0it/s 18.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.9it/s 1.7s
                   all        146          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, can not compute metrics without labels


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:850: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      7/200     0.812G          0     0.4792          0          0        320:  72% ━━━━━━━━╸─── 53/74 3.4it/s 13.4s

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Evaluate YOLOv8 on TEST split only, with data.yaml path provided.
- data.yaml を読み込み、'test' を引数 test_images_dir で上書き
- Ultralytics の model.val() を test のみ実行（mAP50-95, mAP50, Precision, Recall, per-class AP）
- 生成物はすべて save_dir に保存（JSON/図）
"""

from pathlib import Path
import json
from typing import Dict, List, Optional, Any

import numpy as np
import torch
from ultralytics import YOLO


def evaluate_on_test_only(
    weights_path: str,
    data_yaml_path: str,
    save_dir: str,
    batch_size: int = 16,
    conf_threshold: float = 0.001,  # PRの安定化のため低め推奨
    iou: float = 0.50,
    device: Optional[str] = None,
    plots: bool = True,
    save_coco_json: bool = False,
) -> Dict[str, Any]:
    """
    data.yaml を参照しつつ、test split のみ評価を実行し、save_dir に出力をまとめる。

    Args:
        weights_path: 学習済み重み
        data_yaml_path: data.yaml のパス
        test_images_dir: テスト画像フォルダ（直下に *.jpg 等、隣に labels/ があることを想定）
        save_dir: 出力先ディレクトリ（図やJSONをここに保存）
        batch_size: 評価時バッチサイズ
        conf_threshold: 推論の信頼度下限（PRやmAP算出用は低めが一般的）
        iou: IoUしきい値（表示/PR用。mAP50-95 は内部で全IoU平均）
        device: 'cuda'|'mps'|'cpu'（Noneなら自動）
        plots: PR曲線などの図を保存
        save_coco_json: COCO形式jsonの保存（必要な場合のみ True）

    Returns:
        dict: 主要メトリクスと per-class AP を含む辞書
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # ---- デバイス決定 ----
    if device is None:
        if torch.cuda.is_available():
            device = "cuda"
        elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

    print("🔍 Test-only evaluation")
    print(f"  weights: {weights_path}")
    print(f"  data.yaml: {data_yaml_path}")
    print(f"  save_dir: {save_dir}")
    print(f"  device: {device}")

    # ---- data.yaml をロードし、test を上書き ----
    import yaml
    with open(data_yaml_path, "r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)

    # names/nc が yaml 側に無い場合はモデルから補完
    model = YOLO(weights_path)
    try:
        model.to(device)
    except Exception:
        pass

    model_names = model.names
    if isinstance(model_names, dict):
        id2name = model_names
        model_class_names = [id2name[i] for i in sorted(id2name.keys())]
    else:
        model_class_names = list(model_names)

    if "names" not in data_cfg or data_cfg.get("nc") is None:
        data_cfg["names"] = data_cfg.get("names", model_class_names)
        data_cfg["nc"] = data_cfg.get("nc", len(data_cfg["names"]))



    # ---- 評価実行（test のみ）----
    # project/name を指定して、Ultralytics の出力ファイルを save_dir 配下に集約
    # exist_ok=True で毎回上書き可能に
    r = model.val(
        data=data_yaml_path,
        split="test",
        conf=conf_threshold,
        iou=iou,
        batch=batch_size,
        device=device,
        plots=plots,
        save_json=save_coco_json,
        project=str(save_dir),
        name="val",          # 出力は save_dir/val/ にまとまる
        exist_ok=True,
        verbose=False,
    )

    # ---- メトリクス抽出 ----
    # r.box に mAP/Precision/Recall が入る（Ultralytics v8）
    val_report = {
        "map50_95": _to_float_safe(getattr(r.box, "map", None)),       # mAP@[.50:.95]
        "map50": _to_float_safe(getattr(r.box, "map50", None)),        # mAP@0.50
        "precision": _to_float_safe(getattr(r.box, "precision", None)),
        "recall": _to_float_safe(getattr(r.box, "recall", None)),
        "class_names": data_cfg.get("names", model_class_names),
        "speed_ms_per_image": _maybe_speed_ms_per_image(r),
    }

    # per-class AP(0.5:0.95)
    maps = getattr(r.box, "maps", None)
    per_class = []
    if maps is not None:
        maps_list = list(maps)
        for cid, cname in enumerate(val_report["class_names"]):
            ap_c = float(maps_list[cid]) if cid < len(maps_list) and maps_list[cid] is not None else None
            per_class.append({"class_id": cid, "class_name": cname, "AP50_95": ap_c})
    val_report["per_class"] = per_class

    # 追加: results_dict がある場合はそのまま保存（バージョン依存）
    results_dict = getattr(r, "results_dict", None)
    if isinstance(results_dict, dict):
        val_report["results_dict"] = {k: _to_float_safe(v) for k, v in results_dict.items()}

    # JSON保存
    val_json_path = save_dir / "val_report.json"
    with open(val_json_path, "w", encoding="utf-8") as f:
        json.dump(val_report, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved: {val_json_path}")

    # 端末出力（要約）
    print("\n📊 Test-only Evaluation Summary")
    print(f"- mAP50-95: {val_report['map50_95']:.4f}" if val_report["map50_95"] is not None else "- mAP50-95: N/A")
    print(f"- mAP50   : {val_report['map50']:.4f}" if val_report["map50"] is not None else "- mAP50   : N/A")
    print(f"- Precision: {val_report['precision']:.4f}" if val_report["precision"] is not None else "- Precision: N/A")
    print(f"- Recall   : {val_report['recall']:.4f}" if val_report["recall"] is not None else "- Recall   : N/A")
    if val_report["speed_ms_per_image"] is not None:
        print(f"- Speed    : {val_report['speed_ms_per_image']:.2f} ms/img")
    print(f"📁 Artifacts (plots, etc.): {save_dir/'val'}")

    return val_report


# ---------- helpers ----------
def _to_float_safe(x):
    try:
        if x is None:
            return None
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, (list, tuple)) and len(x) == 1:
            return float(x[0])
        return float(x)
    except Exception:
        return None

def _maybe_speed_ms_per_image(r) -> Optional[float]:
    # Ultralytics v8 は r.speed に dict（preprocess, inference, postprocess など）を持つことがある
    spd = getattr(r, "speed", None)
    if isinstance(spd, dict):
        # 合計を1画像あたりで概算
        total = 0.0
        for k in ("preprocess", "inference", "postprocess"):
            v = spd.get(k)
            try:
                total += float(v)
            except Exception:
                pass
        return total
    return None


# ---------- example ----------
evaluate_on_test_only(
    weights_path="/content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/results/furniture_project_3/furniture_yolov8n_20250813_071331_all_data/weights/best.pt",
    data_yaml_path="/content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/data.yaml",
    save_dir="/content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/results/furniture_project_3/furniture_yolov8n_20250813_071331_night_time_data_val",
    batch_size=16,
    conf_threshold=0.001,
    iou=0.50,
    device=None,
    plots=True,
    save_coco_json=False,
)

🔍 Test-only evaluation
  weights: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/results/furniture_project_3/furniture_yolov8n_20250813_071331_all_data/weights/best.pt
  data.yaml: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/data.yaml
  save_dir: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/results/furniture_project_3/furniture_yolov8n_20250813_071331_night_time_data_val
  device: cuda
Ultralytics 8.3.185 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 2.1±2.4 ms, read: 27.0±37.9 MB/s, size: 157.9 KB)


val: Scanning /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/test/labels.cache... 116 images, 0 backgrounds, 0 corrupt: 100%|██████████| 116/116 [00:00<?, ?it/s]

val: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/test/images/2015_03777.png: 4 duplicate labels removed
val: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/test/images/2015_03785.png: 3 duplicate labels removed
val: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/test/images/2015_03793.jpg: 4 duplicate labels removed
val: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/test/images/2015_03796.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/test/images/2015_03808.jpg: 2 duplicate labels removed
val: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/test/images/2015_03809.jpg: 4 duplicate labels removed
val: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/night_time_data/test/images/2015_03818.jpg: 4 duplicate labels removed
val: /content/drive/MyDrive/Colab Noteboo


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.02it/s]


                   all        116        355      0.676      0.445      0.482      0.268
Speed: 0.1ms preprocess, 15.1ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/results/furniture_project_3/furniture_yolov8n_20250813_071331_night_time_data_val/val
💾 Saved: /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/results/furniture_project_3/furniture_yolov8n_20250813_071331_night_time_data_val/val_report.json

📊 Test-only Evaluation Summary
- mAP50-95: 0.2678
- mAP50   : 0.4819
- Precision: N/A
- Recall   : N/A
- Speed    : 16.81 ms/img
📁 Artifacts (plots, etc.): /content/drive/MyDrive/Colab Notebooks/AmazonRoboticsProject/results/furniture_project_3/furniture_yolov8n_20250813_071331_night_time_data_val/val


{'map50_95': 0.2678157407563277,
 'map50': 0.48185973250935477,
 'precision': None,
 'recall': None,
 'class_names': ['Chair', 'Table'],
 'speed_ms_per_image': 16.806048387938343,
 'per_class': [{'class_id': 0,
   'class_name': 'Chair',
   'AP50_95': 0.30623717433746583},
  {'class_id': 1, 'class_name': 'Table', 'AP50_95': 0.2293943071751897}],
 'results_dict': {'metrics/precision(B)': 0.675878176375832,
  'metrics/recall(B)': 0.4450801877319197,
  'metrics/mAP50(B)': 0.48185973250935477,
  'metrics/mAP50-95(B)': 0.2678157407563277,
  'fitness': 0.2892201399316304}}